# Add fire-year history and last-fire information

This notebook combines a pre-existing pair table that already contains fire-frequency information with annual fire-year rasters, then keeps only pairs whose pre and post footprints share the same historical fire regime before the event of interest.

The workflow is:
1. Load the pair table with fire frequency.
2. Buffer pre and post footprints for raster masking.
3. Scan all fire-year raster tiles and collect the years that burned each footprint.
4. Filter the fire-year histories to years before the identified pair fire date.
5. Keep only pairs whose pre and post footprints have matching fire frequency and matching pre-fire fire-year histories.
6. Estimate the last fire before the pre-fire GEDI observation and compute pre-fire vegetation age.
7. Save the filtered dataset.


In [ ]:
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely import wkt
from shapely.geometry import box, mapping
from tqdm.auto import tqdm


In [ ]:
pairs_path = "GEDI_Pairs/GEDI_Footprint_Pairs_Fire_with_FF.gpkg"
tiles_dir = Path("Fire History/Fire Years")
out_path = "GEDI_Pairs/GEDI_Footprint_Pairs_Fire_FF_FYs.gpkg"
buffer_size = 10


In [ ]:
filter_table = pd.DataFrame(
    [
        {"stage": "Input filter", "filter": "pair source", "value": pairs_path},
        {"stage": "Input filter", "filter": "fire frequency provenance", "value": "use the pre-existing fire-frequency values already stored in the input pair file"},
        {"stage": "Input filter", "filter": "fire year source", "value": f"all .tif files in {tiles_dir}"},
        {"stage": "Buffer filter", "filter": "buffer size", "value": f"{buffer_size} m"},
        {"stage": "Buffer filter", "filter": "buffer CRS", "value": "buffer by pairing_crs, then convert back to EPSG:4326"},
        {"stage": "Buffer filter", "filter": "missing pairing_crs", "value": "skip groups with missing pairing_crs"},
        {"stage": "Tile selection", "filter": "raster tile overlap", "value": "keep only footprints intersecting each raster tile bounds"},
        {"stage": "Raster mask", "filter": "mask mode", "value": "crop=True, filled=False, all_touched=True"},
        {"stage": "Raster value filter", "filter": "nodata removal", "value": "drop nodata values"},
        {"stage": "Raster value filter", "filter": "NaN removal", "value": "drop NaN values"},
        {"stage": "Raster value filter", "filter": "positive year rule", "value": "keep only raster values > 0"},
        {"stage": "Fire history filter", "filter": "years before fire_date", "value": "keep only years strictly earlier than fire_date year"},
        {"stage": "Pair consistency filter", "filter": "same fire frequency", "value": "pre_FF == post_FF, with nulls treated as -9999"},
        {"stage": "Pair consistency filter", "filter": "same pre-fire fire history", "value": "pre_fire_years_before == post_fire_years_before"},
        {"stage": "Last-fire filter", "filter": "last fire before time_1", "value": "max fire year < time_1 year and < fire_date year"},
        {"stage": "Vegetation age filter", "filter": "negative ages", "value": "set pre_veg_age < 0 to NaN"},
        {"stage": "Completeness rule", "filter": "missing fire_date year in fire_years", "value": "append fire_date year when absent"},
    ]
)

display(filter_table)


In [ ]:
pairs = gpd.read_file(pairs_path)
pairs["pair_uid"] = pairs.index.astype(int)
pairs["pre_geom"] = pairs["pre_geom"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
pairs["post_geom"] = pairs["post_geom"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)

print("Loaded pairs:", len(pairs))


In [ ]:
pairs["pre_buff_10"] = None
pairs["post_buff_10"] = None

for pcrs, idxs in tqdm(pairs.groupby("pairing_crs").groups.items(), desc="Buffering by pairing_crs"):
    if pd.isna(pcrs):
        continue
    pre_ll = gpd.GeoSeries(pairs.loc[idxs, "pre_geom"], crs="EPSG:4326")
    post_ll = gpd.GeoSeries(pairs.loc[idxs, "post_geom"], crs="EPSG:4326")
    pairs.loc[idxs, "pre_buff_10"] = pre_ll.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326").values
    pairs.loc[idxs, "post_buff_10"] = post_ll.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326").values


In [ ]:
pre_gdf = gpd.GeoDataFrame(pairs[["pair_uid"]].copy(), geometry=pairs["pre_buff_10"], crs="EPSG:4326")
pre_gdf["footprint"] = "pre"

post_gdf = gpd.GeoDataFrame(pairs[["pair_uid"]].copy(), geometry=pairs["post_buff_10"], crs="EPSG:4326")
post_gdf["footprint"] = "post"

footprints = pd.concat([pre_gdf, post_gdf], ignore_index=True)
footprints = gpd.GeoDataFrame(footprints, geometry="geometry", crs="EPSG:4326")
footprints = footprints[footprints.geometry.notna() & ~footprints.geometry.is_empty].copy()

print("Footprint rows:", len(footprints))


In [ ]:
tile_paths = sorted(tiles_dir.glob("*.tif"))
years_acc = defaultdict(set)
fp_cache = {}

for tile in tqdm(tile_paths, desc="Processing fire-year tiles"):
    with rasterio.open(tile) as ds:
        crs_key = str(ds.crs)
        if crs_key not in fp_cache:
            fp_cache[crs_key] = footprints.to_crs(ds.crs) if footprints.crs != ds.crs else footprints.copy()
            fp_cache[crs_key].sindex

        fp = fp_cache[crs_key]
        tile_poly = box(*ds.bounds)
        idx = list(fp.sindex.query(tile_poly, predicate="intersects"))
        if not idx:
            continue

        subset = fp.iloc[idx]
        for row in subset.itertuples(index=False):
            try:
                out, _ = mask(ds, [mapping(row.geometry)], crop=True, filled=False, all_touched=True)
            except Exception:
                continue

            vals = out.compressed() if np.ma.isMaskedArray(out) else out.ravel()
            if ds.nodata is not None:
                vals = vals[vals != ds.nodata]
            vals = vals[~np.isnan(vals)]
            vals = vals[vals > 0]
            if vals.size == 0:
                continue

            years_acc[(int(row.pair_uid), row.footprint)].update(vals.astype(int).tolist())


In [ ]:
records = []
for (pid, fp), ys in years_acc.items():
    ys = sorted(ys)
    records.append({
        "pair_uid": pid,
        "footprint": fp,
        "fire_years": ys,
        "fire_years_str": ",".join(map(str, ys)),
    })

fire_years_long = pd.DataFrame(records)
pre_map = fire_years_long.loc[fire_years_long["footprint"] == "pre"].set_index("pair_uid")["fire_years_str"].to_dict()
post_map = fire_years_long.loc[fire_years_long["footprint"] == "post"].set_index("pair_uid")["fire_years_str"].to_dict()

pairs["pre_fire_years"] = pairs["pair_uid"].map(pre_map).fillna("")
pairs["post_fire_years"] = pairs["pair_uid"].map(post_map).fillna("")

print("Pairs with any fire history:", ((pairs["pre_fire_years"] != "") | (pairs["post_fire_years"] != "")).sum())


In [ ]:
pairs["fire_date_dt"] = pd.to_datetime(pairs["fire_date"], errors="coerce")
pairs["fire_date_year"] = pairs["fire_date_dt"].dt.year

def years_before_cutoff(years_value, cutoff_year):
    if pd.isna(cutoff_year):
        return ""
    cutoff_year = int(cutoff_year)
    if years_value is None or (isinstance(years_value, float) and np.isnan(years_value)):
        return ""
    s = str(years_value).strip()
    if s == "":
        return ""
    years = [int(token.strip()) for token in s.split(",") if token.strip().isdigit()]
    kept = sorted(set(y for y in years if y < cutoff_year))
    return ",".join(map(str, kept))

pairs["pre_fire_years_before"] = [years_before_cutoff(y, c) for y, c in zip(pairs["pre_fire_years"], pairs["fire_date_year"])]
pairs["post_fire_years_before"] = [years_before_cutoff(y, c) for y, c in zip(pairs["post_fire_years"], pairs["fire_date_year"])]

mask_same = (
    pairs["pre_FF"].fillna(-9999).eq(pairs["post_FF"].fillna(-9999))
    & pairs["pre_fire_years_before"].fillna("").eq(pairs["post_fire_years_before"].fillna(""))
)
pairs_final = pairs.loc[mask_same].copy()

pairs_final["fire_frequency"] = pairs_final["pre_FF"]
pairs_final["fire_years"] = pairs_final["pre_fire_years"]
pairs_final["fire_years_before_fire_date"] = pairs_final["pre_fire_years_before"]
pairs_final = pairs_final.drop(columns=[
    "pre_FF",
    "post_FF",
    "pre_fire_years",
    "post_fire_years",
    "pre_fire_years_before",
    "post_fire_years_before",
], errors="ignore")

print("Rows kept after consistency filter:", len(pairs_final))


In [ ]:
pairs_final["time_1_dt"] = pd.to_datetime(pairs_final["time_1"], errors="coerce")

def last_fire_before_time1(fire_years_str, time1_dt, fire_date_dt):
    if pd.isna(time1_dt):
        return np.nan
    s = "" if pd.isna(fire_years_str) else str(fire_years_str).strip()
    if s == "":
        return np.nan
    years = [int(token.strip()) for token in s.split(",") if token.strip().isdigit()]
    valid = [y for y in years if y < time1_dt.year]
    if pd.notna(fire_date_dt):
        valid = [y for y in valid if y < fire_date_dt.year]
    return max(valid) if valid else np.nan

pairs_final["last_fire_year_for_time1"] = [
    last_fire_before_time1(fy, t1, fd)
    for fy, t1, fd in zip(pairs_final["fire_years"], pairs_final["time_1_dt"], pairs_final["fire_date_dt"])
]
pairs_final["pre_veg_age"] = pairs_final["time_1_dt"].dt.year - pairs_final["last_fire_year_for_time1"]
pairs_final.loc[pairs_final["pre_veg_age"] < 0, "pre_veg_age"] = np.nan


In [ ]:
def parse_years(s):
    if pd.isna(s):
        return []
    s = str(s).strip()
    if s == "":
        return []
    return sorted(set(int(token.strip()) for token in s.split(",") if token.strip().isdigit()))

def ensure_fire_date_year_in_list(fire_years_str, fire_date_year):
    ys = parse_years(fire_years_str)
    if pd.notna(fire_date_year):
        ys = sorted(set(ys + [int(fire_date_year)]))
    return ",".join(map(str, ys))

years_lists = pairs_final["fire_years"].apply(parse_years)
has_fire_date_year = [
    (not pd.isna(fd)) and (int(fd) in ys)
    for fd, ys in zip(pairs_final["fire_date_year"], years_lists)
]
missing_mask = ~pd.Series(has_fire_date_year, index=pairs_final.index)

pairs_final.loc[missing_mask, "fire_years"] = [
    ensure_fire_date_year_in_list(fy, fd)
    for fy, fd in zip(
        pairs_final.loc[missing_mask, "fire_years"],
        pairs_final.loc[missing_mask, "fire_date_year"],
    )
]

print("Rows updated to include fire_date year:", int(missing_mask.sum()))


In [ ]:
pairs_out = pairs_final.drop(columns=[
    "pre_buff_10",
    "post_buff_10",
    "time_1_dt",
    "last_fire_year_for_time1",
], errors="ignore").copy()

pairs_out["pre_geom"] = gpd.GeoSeries(pairs_out["pre_geom"]).to_wkt()
pairs_out["post_geom"] = gpd.GeoSeries(pairs_out["post_geom"]).to_wkt()
pairs_out.to_file(out_path, driver="GPKG")

print("Pairs with primary forest before fire:", int(pairs_out["pre_veg_age"].isna().sum()))
print("Saved:", out_path)
